# Silver Validation

Validate and deduplicate Bronze employee data, then save clean, rejected and excluded records as Delta tables.

Source: `employees_pipeline.bronze.employees_bronze`

Target: `employees_pipeline.silver`


## 1. Load Bronze data


In [0]:
bronze_df = spark.table(
    "employees_pipeline.bronze.employees_bronze"
)

display(bronze_df)

In [0]:
bronze_df.printSchema()

In [0]:
bronze_df.count()

## 2. Cast numeric columns


In [0]:
from pyspark.sql.functions import col

silver_df = (
    bronze_df
    .withColumn("employee_id", col("employee_id").cast("int"))
    .withColumn("department_id", col("department_id").cast("int"))
    .withColumn("salary", col("salary").cast("int"))
)

In [0]:
silver_df.select(
    "employee_id",
    "department_id",
    "salary"
).printSchema()

## 3. Parse dates


In [0]:
silver_df.select("hire_date").show(500, truncate=False)

In [0]:
from pyspark.sql.functions import try_to_timestamp, lit

silver_df = silver_df.withColumn(
    "hire_date_parsed",
    try_to_timestamp(
        col("hire_date"),
        lit("yyyy-MM-dd HH:mm:ss")
    )
)

In [0]:
silver_df.select(
    "hire_date",
    "hire_date_parsed"
).show(50, truncate=False)

In [0]:
silver_df = silver_df.withColumn(
    "updated_at",
    try_to_timestamp(
        col("updated_at"),
        lit("yyyy-MM-dd HH:mm:ss")
    )
)

In [0]:
silver_df.select("updated_at").show(100, truncate=False)

In [0]:
silver_df.select(
    "updated_at"
).printSchema()

## 4. Validate employee data


In [0]:

from pyspark.sql.functions import when

silver_df = silver_df.withColumn(
    "salary_error",
    when(
        col("salary").isNull()
        | (col("salary") <= 0),
        "INVALID_SALARY"
    ).otherwise(None)
)


In [0]:
silver_df.select(
    "employee_id",
    "salary",
    "salary_error"
).show(20, truncate=False)

In [0]:
silver_df = silver_df.withColumn(
    "status_error",
    when(
        col("status").isNull()
        | ~col("status").isin("ACTIVE", "INACTIVE"),
        "INVALID_STATUS"
    ).otherwise(None)
)

In [0]:
display(silver_df)

In [0]:
silver_df.select(
    "employee_id",
    "status",
    "status_error"
).show(30, truncate=False)

In [0]:
from pyspark.sql.functions import when, trim

silver_df = silver_df.withColumn(
    "country_error",
    when(
        col("country").isNull()
        | (trim(col("country")) == ""),
        "EMPTY_COUNTRY"
    ).otherwise(None)
)

In [0]:
silver_df.select(
    "employee_id",
    "country",
    "country_error"
).show(50, truncate=False)

In [0]:
email_pattern = r"\A[^@\s]+@[^@\s]+\.[^@\s]+\z"

In [0]:
silver_df = silver_df.withColumn(
    "email_error",
    when(
        col("email").isNull()
        | (trim(col("email")) == ""),
        "EMPTY_EMAIL"
    )
    .when(
        ~col("email").rlike(email_pattern),
        "INVALID_EMAIL"
    )
    .otherwise(None)
)

In [0]:
display(silver_df)

In [0]:


silver_df = silver_df.withColumn(
    "hire_date_error",
    when(
        col("hire_date").isNull()
        | (col("hire_date") == ""),
        "EMPTY_HIRE_DATE"
    ).when(
        col("hire_date_parsed").isNull(),
        "INVALID_HIRE_DATE"
    )
)



In [0]:
print(silver_df.columns)

In [0]:
silver_df.select(
    "hire_date_error"
).filter(
    col("hire_date_error").isNotNull()
).show(50, truncate=False)

In [0]:
silver_df.groupBy("hire_date_error").count().show()


In [0]:
silver_df.filter(
    col("hire_date_error").isNotNull()
).groupBy(
    "hire_date_error"
).count().show()

### Additional validation rules

Check employee IDs, department IDs, employment types and update timestamps before deduplication.


In [0]:
silver_df = (
    silver_df
    .withColumn(
        "employee_id_error",
        when(
            col("employee_id").isNull() | (col("employee_id") <= 0),
            "INVALID_EMPLOYEE_ID"
        )
    )
    .withColumn(
        "department_id_error",
        when(
            col("department_id").isNull() |
            ~col("department_id").isin(10, 20, 30, 40, 50),
            "INVALID_DEPARTMENT_ID"
        )
    )
    .withColumn(
        "employment_type_error",
        when(
            col("employment_type").isNull() |
            ~col("employment_type").isin("FULL_TIME", "PART_TIME", "CONTRACTOR"),
            "INVALID_EMPLOYMENT_TYPE"
        )
    )
    .withColumn(
        "updated_at_error",
        when(
            col("updated_at").isNull(),
            "INVALID_UPDATED_AT"
        )
    )
)


## 5. Keep the latest employee record


In [0]:
duplicates_df = silver_df.groupBy("employee_id").count().filter(
    col("count") > 1
)

duplicates_df.show()


In [0]:
duplicates_df.count()

In [0]:
silver_df.printSchema()


In [0]:

from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

employee_window = (
    Window.partitionBy("employee_id")
    .orderBy(
        col("updated_at").desc_nulls_last(),
        col("_ingested_at").desc_nulls_last()
    )
)


In [0]:
silver_df = silver_df.withColumn(
    "row_num",
    row_number().over(employee_window)
)

In [0]:
latest_df = silver_df.filter(
    col("row_num") == 1
)


In [0]:
excluded_df = silver_df.filter(
    col("row_num") > 1
)

print(excluded_df.count())
excluded_df.show()

In [0]:
print("Wszystkie rekordy:", silver_df.count())
print("Najnowsze rekordy:", latest_df.count())
print("Starsze duplikaty:", excluded_df.count())


## 6. Collect validation errors


In [0]:

from pyspark.sql.functions import concat_ws

latest_df = latest_df.withColumn(
    "error_reasons",
    concat_ws(
        ", ",
        col("salary_error"),
        col("status_error"),
        col("country_error"),
        col("email_error"),
        col("hire_date_error"),
        col("employee_id_error"),
        col("department_id_error"),
        col("employment_type_error"),
        col("updated_at_error")
    )
)


In [0]:
latest_df.select(
    "employee_id",
    "salary_error",
    "status_error",
    "country_error",
    "email_error",
    "hire_date_error",
    "employee_id_error",
    "department_id_error",
    "employment_type_error",
    "updated_at_error",
    "error_reasons"
).filter(
    col("error_reasons") != ""
).show(50, truncate=False)


## 7. Split clean and rejected records


In [0]:
clean_df = latest_df.filter(
    col("error_reasons") == ""
)


In [0]:
display(clean_df)

In [0]:
rejected_df = latest_df.filter(
    col("error_reasons") != ""
)

In [0]:
display(rejected_df)

## 8. Check record counts


In [0]:
source_count = silver_df.count()
latest_count = latest_df.count()
excluded_count = excluded_df.count()
clean_count = clean_df.count()
rejected_count = rejected_df.count()

print("Source records:", source_count)
print("Latest records:", latest_count)
print("Older duplicates:", excluded_count)
print("Clean:", clean_count)
print("Rejected:", rejected_count)

print("Deduplication balance:", latest_count + excluded_count == source_count)
print("Silver balance:", clean_count + rejected_count == latest_count)
print("Final balance:", clean_count + rejected_count + excluded_count == source_count)


## 9. Prepare the clean dataset


In [0]:

clean_df = clean_df.drop(
    "row_num",
    "salary_error",
    "status_error",
    "country_error",
    "email_error",
    "hire_date_error",
    "employee_id_error",
    "department_id_error",
    "employment_type_error",
    "updated_at_error",
    "error_reasons"
)


In [0]:
from pyspark.sql.functions import to_date

clean_df = clean_df.withColumn(
    "hire_date",
    to_date(col("hire_date_parsed"))
)

In [0]:
clean_df.select(
    "hire_date",
    "hire_date_parsed"
).printSchema()

In [0]:
clean_df.printSchema()


In [0]:
clean_df = clean_df.drop(
    "hire_date_parsed"
)


In [0]:
clean_df.printSchema()


In [0]:
print(clean_df.count())


## 10. Save Silver Delta tables


In [0]:
(
    clean_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("employees_pipeline.silver.employees_clean")
)

In [0]:
display(
    spark.table("employees_pipeline.silver.employees_clean")
)


In [0]:
spark.table(
    "employees_pipeline.silver.employees_clean"
).printSchema()

In [0]:
display(rejected_df)

In [0]:
rejected_df = rejected_df.drop(
    "row_num"
)


In [0]:
rejected_df.printSchema()

In [0]:

(
    rejected_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("employees_pipeline.silver.employees_rejected")
)


In [0]:
print(
    spark.table("employees_pipeline.silver.employees_rejected").count()
)


In [0]:
display(excluded_df)


In [0]:
excluded_df = excluded_df.withColumn(
    "exclusion_reason",
    lit("OLDER_DUPLICATE")
)


In [0]:
excluded_df.select(
    "employee_id",
    "row_num",
    "exclusion_reason"
).show()

In [0]:
(
    excluded_df.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .saveAsTable("employees_pipeline.silver.employees_excluded")
)


## 11. Verify saved tables


In [0]:
clean_table_count = spark.table("employees_pipeline.silver.employees_clean").count()
rejected_table_count = spark.table("employees_pipeline.silver.employees_rejected").count()
excluded_table_count = spark.table("employees_pipeline.silver.employees_excluded").count()

total_count = (
    clean_table_count
    + rejected_table_count
    + excluded_table_count
)

print("Clean:", clean_table_count)
print("Rejected:", rejected_table_count)
print("Excluded:", excluded_table_count)
print("Total:", total_count)
print("Final balance correct:", total_count == source_count)